# Tapi · Exploración inicial con SQL

Ejecutá las celdas en orden. Usamos DuckDB para consultar los CSV con SQL dentro de Colab.

La celda de carga descarga los tres CSV desde GitHub y crea las tablas cuentas, consultas y pagos.


In [ ]:
%pip -q install duckdb==1.4.3

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import duckdb

url_base = "https://raw.githubusercontent.com/DaianaSep/tapi_Challenge/main"
archivos = {
    "cuentas": "B_agenda_cuentas.csv",
    "consultas": "B_agenda_consultas.csv",
    "pagos": "B_agenda_pagos.csv",
}

carpeta = Path("data")
carpeta.mkdir(exist_ok=True)

nueva_conexion = duckdb.connect()
try:
    for tabla, archivo in archivos.items():
        destino = carpeta / archivo
        urlretrieve(f"{url_base}/{archivo}", destino)
        nueva_conexion.execute(
            f"CREATE TABLE {tabla} AS "
            "SELECT * FROM read_csv(?, header=true, all_varchar=true)",
            [str(destino)],
        )

except Exception:
    nueva_conexion.close()
    raise

if "conexion" in globals():
    conexion.close()
conexion = nueva_conexion

def consultar(sql):
    return conexion.execute(sql).df()

print("CSV descargados desde GitHub. Tablas cuentas, consultas y pagos listas.")


Seleccioná los tres archivos CSV del challenge.


TypeError: 'NoneType' object is not subscriptable

Cargamos inicialmente todas las columnas como texto para conservar IDs y timestamps originales. Los campos vacíos se leen como NULL. Más adelante convertiremos fechas y montos para los cálculos que lo requieran.

## Muestra de cuentas

Cada fila representa una adhesión a la Agenda. LIMIT devuelve hasta diez filas. Sin ORDER BY no hay un orden garantizado.

In [ ]:
consultar("""
SELECT *
FROM cuentas
LIMIT 10
""")

## Muestra de consultas

Cada fila representa un intento de consulta. LIMIT devuelve hasta diez filas. Sin ORDER BY no hay un orden garantizado.

In [ ]:
consultar("""
SELECT *
FROM consultas
LIMIT 10
""")

## Muestra de pagos

Cada fila representa un pago realizado. LIMIT devuelve hasta diez filas. Sin ORDER BY no hay un orden garantizado.

In [ ]:
consultar("""
SELECT *
FROM pagos
LIMIT 10
""")

## Cantidad de registros

COUNT(*) cuenta filas. UNION ALL apila los resultados de las tres tablas.

In [ ]:
consultar("""
SELECT 'cuentas' AS tabla, COUNT(*) AS filas FROM cuentas
UNION ALL
SELECT 'consultas' AS tabla, COUNT(*) AS filas FROM consultas
UNION ALL
SELECT 'pagos' AS tabla, COUNT(*) AS filas FROM pagos
""")

## Identificadores de cuentas

Compará filas, IDs informados e IDs distintos. COUNT(columna) excluye NULL. Una diferencia requiere investigar faltantes o repeticiones.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(account_id) AS filas_con_id,
    COUNT(DISTINCT account_id) AS cuentas_distintas
FROM cuentas
""")

## Identificadores de consultas

Compará filas, IDs informados e IDs distintos. COUNT(columna) excluye NULL. Una diferencia requiere investigar faltantes o repeticiones.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(consulta_id) AS filas_con_id,
    COUNT(DISTINCT consulta_id) AS consultas_distintas
FROM consultas
""")

## Identificadores de pagos

Compará filas, IDs informados e IDs distintos. COUNT(columna) excluye NULL. Una diferencia requiere investigar faltantes o repeticiones.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(pago_id) AS filas_con_id,
    COUNT(DISTINCT pago_id) AS pagos_distintos
FROM pagos
""")

## Convenios y proveedores

Una fila por combinación de convenio y proveedor. Permite identificar convenios compartidos y comparar el volumen de consultas.

In [ ]:
consultar("""
SELECT
    convenio,
    proveedor,
    COUNT(*) AS cantidad_consultas
FROM consultas
GROUP BY convenio, proveedor
ORDER BY convenio, proveedor
""")

## Tus observaciones

Anotá qué encontraste en cada consulta y qué querés investigar después. Revisaremos origen, resultado, fechas y campos vacíos en el próximo paso.

# Paso 2 · Validar los datos antes del ratio semanal

## Origen y resultado

Cada fila del resultado cuenta intentos de consulta. Una cuenta puede devolver `deuda_encontrada` varias veces.

In [ ]:
consultar("""
SELECT origen, resultado, COUNT(*) AS intentos
FROM consultas
GROUP BY origen, resultado
ORDER BY origen, intentos DESC
""")

## Fechas cubiertas

Los timestamps vienen en formato ISO. Tomamos los primeros diez caracteres para conservar el día local registrado en el CSV.

In [ ]:
consultar("""
SELECT 'cuentas' AS tabla, MIN(fecha_alta) AS desde, MAX(fecha_alta) AS hasta
FROM cuentas
UNION ALL
SELECT 'consultas', MIN(SUBSTR("timestamp", 1, 10)), MAX(SUBSTR("timestamp", 1, 10))
FROM consultas
UNION ALL
SELECT 'pagos', MIN(SUBSTR(fecha_pago, 1, 10)), MAX(SUBSTR(fecha_pago, 1, 10))
FROM pagos
""")

## Campos vacíos según resultado

En consultas sin deuda, es esperable que monto y vencimiento estén vacíos. Revisá en especial la fila de `deuda_encontrada`.

In [ ]:
consultar("""
SELECT
    resultado,
    COUNT(*) AS intentos,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(monto), '') IS NULL) AS sin_monto,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(fecha_vencimiento), '') IS NULL) AS sin_vencimiento
FROM consultas
GROUP BY resultado
ORDER BY resultado
""")

## Paso 3 · Ratio semanal de DeltaPay

Dividimos consultas por pagos de los convenios de DeltaPay. Las semanas van de lunes a domingo, usando la fecha local del CSV. Marcamos los extremos parciales y dejamos indefinido el ratio cuando hay cero pagos. El exceso supone que se mantienen los pagos observados.

In [ ]:
consultar("""
WITH convenios_delta AS (
    SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay'
),
consultas_semanales AS (
    SELECT date_trunc('week', CAST(substr("timestamp", 1, 10) AS DATE)) AS semana,
           COUNT(*) AS consultas
    FROM consultas
    WHERE proveedor = 'deltapay'
    GROUP BY 1
),
pagos_semanales AS (
    SELECT date_trunc('week', CAST(substr(fecha_pago, 1, 10) AS DATE)) AS semana,
           COUNT(*) AS pagos
    FROM pagos
    WHERE convenio IN (SELECT convenio FROM convenios_delta)
    GROUP BY 1
)
SELECT semana, COALESCE(consultas, 0) AS consultas, COALESCE(pagos, 0) AS pagos,
       ROUND(COALESCE(consultas, 0) * 1.0 / NULLIF(pagos, 0), 2) AS consultas_por_pago,
       CASE WHEN pagos > 0 THEN GREATEST(COALESCE(consultas, 0) - 6 * pagos, 0) END AS exceso,
       semana < DATE '2026-08-01' OR semana + INTERVAL '6 days' > DATE '2026-10-31' AS semana_parcial
FROM consultas_semanales
FULL OUTER JOIN pagos_semanales USING (semana)
ORDER BY semana
""")

## Hipótesis · Guardar una deuda ya detectada

Usamos cuenta, convenio, monto y vencimiento como aproximación a una factura. Contamos las detecciones posteriores de esa combinación si no hubo un pago registrado entre ambas. Es un conteo histórico de repeticiones, no una estimación directa de ahorro. Una regla operativa necesitaría invalidar al pagar y refrescar el dato cuando pueda cambiar.

In [ ]:
consultar("""
WITH detecciones AS (
    SELECT consulta_id, account_id, convenio, monto, fecha_vencimiento, "timestamp",
           LAG("timestamp") OVER (
               PARTITION BY account_id, convenio, monto, fecha_vencimiento
               ORDER BY "timestamp", consulta_id
           ) AS deteccion_anterior
    FROM consultas
    WHERE proveedor = 'deltapay' AND resultado = 'deuda_encontrada'
      AND monto IS NOT NULL AND fecha_vencimiento IS NOT NULL
)
SELECT convenio,
       COUNT(*) AS consultas_con_deuda,
       COUNT(*) FILTER (
           WHERE deteccion_anterior IS NOT NULL
             AND NOT EXISTS (
                 SELECT 1 FROM pagos p
                 WHERE p.account_id = d.account_id AND p.convenio = d.convenio
                   AND p.monto = d.monto AND p.fecha_vencimiento = d.fecha_vencimiento
                   AND p.fecha_pago >= d.deteccion_anterior
                   AND p.fecha_pago <= d."timestamp"
             )
       ) AS detecciones_repetidas_sin_pago_intermedio
FROM detecciones d
GROUP BY convenio
ORDER BY detecciones_repetidas_sin_pago_intermedio DESC
""")

## Paso 4 · Simular la reutilización de una deuda

Probamos una vigencia de 6, 12 y 24 horas para la última deuda detectada por cuenta y convenio. Volvemos a consultar si la vigencia terminó, si llegó el día del vencimiento o si se registró un pago posterior a la detección para esa cuenta y convenio. Cualquier pago invalida la deuda guardada para esa cuenta. Las consultas omitidas no renuevan la vigencia.

La decisión solo usa eventos conocidos hasta la hora de la consulta. Usamos la respuesta histórica de una consulta omitida para medir qué información se habría perdido. Un cambio de monto o vencimiento cuenta como deuda distinta. El ratio estimado conserva los pagos históricos y no modela cómo podrían cambiar si Tapi detectara una deuda más tarde. El resumen usa las 12 semanas completas.

In [ ]:
import pandas as pd

consultas_delta = consultar("""
    SELECT account_id, convenio, "timestamp", origen, resultado, monto, fecha_vencimiento
    FROM consultas
    WHERE proveedor = 'deltapay'
    ORDER BY "timestamp", consulta_id
""")
pagos_delta = consultar("""
    SELECT account_id, convenio, fecha_pago
    FROM pagos
    WHERE convenio IN (
        SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay'
    )
""")

# Los CSV usan un único desplazamiento horario (-06:00).
from datetime import date, datetime

consultas_delta["timestamp"] = consultas_delta["timestamp"].map(datetime.fromisoformat)
consultas_delta["fecha_vencimiento"] = consultas_delta["fecha_vencimiento"].map(
    lambda valor: date.fromisoformat(valor) if valor is not None else None
)
pagos_delta["fecha_pago"] = pagos_delta["fecha_pago"].map(datetime.fromisoformat)
eventos = list(consultas_delta.itertuples(index=False))
pagos_observados = list(pagos_delta.itertuples(index=False))


In [ ]:
from bisect import bisect_right
from collections import Counter, defaultdict
from datetime import timedelta

def simular_cache(consultas, pagos, horas, auditoria=None, intervalo_programado=None):
    pagos_por_cuenta = defaultdict(list)
    for pago in pagos:
        pagos_por_cuenta[pago.account_id, pago.convenio].append(pago.fecha_pago)
    for fechas in pagos_por_cuenta.values():
        fechas.sort()

    if isinstance(intervalo_programado, timedelta):
        intervalo_programado = dict.fromkeys(('CONV-TV', 'CONV-AGUA'), intervalo_programado)
    vigencia = timedelta(hours=horas)
    cache = {}
    ultima_programada = {}
    semanas = defaultdict(Counter)

    for consulta in consultas:
        clave = (consulta.account_id, consulta.convenio)
        momento = consulta.timestamp
        dia = momento.date()
        semana = dia - timedelta(days=dia.weekday())
        metricas = semanas[semana]
        metricas["consultas"] += 1

        if auditoria is not None and intervalo_programado is not None:
            if consulta.resultado == "deuda_encontrada":
                factura = (*clave, consulta.fecha_vencimiento)
                auditoria["primera_observada"].setdefault(factura, momento)

        anterior = cache.get(clave)
        if anterior is not None:
            detectada, monto, vencimiento = anterior
            fechas_pago = pagos_por_cuenta[clave]
            siguiente_pago = bisect_right(fechas_pago, detectada)
            hubo_pago = (
                siguiente_pago < len(fechas_pago)
                and fechas_pago[siguiente_pago] <= momento
            )
            if momento - detectada < vigencia and dia < vencimiento and not hubo_pago:
                metricas["omitidas"] += 1
                if consulta.resultado == "deuda_encontrada":
                    if consulta.monto == monto and consulta.fecha_vencimiento == vencimiento:
                        metricas["misma_deuda"] += 1
                    else:
                        metricas["deuda_distinta"] += 1
                        if auditoria is not None:
                            auditoria["cambios_omitidos"].append(consulta)
                        if consulta.fecha_vencimiento is not None and consulta.fecha_vencimiento <= dia:
                            metricas["distinta_vence_hoy_o_antes"] += 1
                else:
                    metricas[consulta.resultado] += 1
                continue

        if (intervalo_programado is not None and consulta.origen == "programada"
            and consulta.convenio in {"CONV-TV", "CONV-AGUA"}):
            ultima = ultima_programada.get(clave)
            if ultima is not None and momento - ultima < intervalo_programado.get(consulta.convenio, timedelta(0)):
                metricas["omitidas"] += 1
                continue
            ultima_programada[clave] = momento

        if auditoria is not None and consulta.resultado == "deuda_encontrada":
            if intervalo_programado is not None:
                factura = (*clave, consulta.fecha_vencimiento)
                auditoria["primera_con_regla"].setdefault(factura, momento)
            auditoria["detecciones_ejecutadas"].append(consulta)
        cache.pop(clave, None)
        if (consulta.resultado == "deuda_encontrada"
            and consulta.monto is not None and consulta.fecha_vencimiento is not None):
            cache[clave] = (momento, consulta.monto, consulta.fecha_vencimiento)

    for pago in pagos:
        dia = pago.fecha_pago.date()
        semanas[dia - timedelta(days=dia.weekday())]["pagos"] += 1

    return semanas


In [ ]:
filas = []
for horas in (6, 12, 24):
    for semana, valores in sorted(simular_cache(eventos, pagos_observados, horas).items()):
        pagos_semana = valores["pagos"]
        consultas_estimadas = valores["consultas"] - valores["omitidas"]
        filas.append({
            "horas": horas,
            "semana": semana,
            "parcial": semana < date(2026, 8, 1) or semana + timedelta(days=6) > date(2026, 10, 31),
            "consultas": valores["consultas"],
            "pagos": pagos_semana,
            "omitidas": valores["omitidas"],
            "ratio_observado": valores["consultas"] / pagos_semana if pagos_semana else None,
            "ratio_estimado": consultas_estimadas / pagos_semana if pagos_semana else None,
            **{clave: valores[clave] for clave in (
                "misma_deuda", "deuda_distinta", "distinta_vence_hoy_o_antes",
                "sin_deuda", "timeout", "error"
            )},
        })

detalle_cache = pd.DataFrame(filas)
completas = detalle_cache.loc[~detalle_cache["parcial"]]
resumen_cache = completas.groupby("horas").agg(
    consultas=("consultas", "sum"),
    omitidas=("omitidas", "sum"),
    misma_deuda=("misma_deuda", "sum"),
    deuda_distinta=("deuda_distinta", "sum"),
    distinta_vence_hoy_o_antes=("distinta_vence_hoy_o_antes", "sum"),
    sin_deuda=("sin_deuda", "sum"),
    timeout=("timeout", "sum"),
    error=("error", "sum"),
    semanas_dentro_del_limite=("ratio_estimado", lambda ratios: (ratios <= 6).sum()),
    menor_ratio_estimado=("ratio_estimado", "min"),
)
resumen_cache["reduccion_pct"] = 100 * resumen_cache["omitidas"] / resumen_cache["consultas"]
display(resumen_cache.round(2))
display(detalle_cache.round(2))


## Paso 5 · Revisar el riesgo de guardar la deuda 12 horas

Por cada respuesta omitida con otro monto o vencimiento, buscamos cuándo una consulta que sí se habría ejecutado devolvió esa misma combinación de cuenta, convenio, monto y vencimiento. Medimos la demora desde la consulta omitida y el margen en días calendario. Una detección el día del vencimiento requiere conocer la hora de corte. Si no volvemos a ver esa combinación en el histórico, la demora queda indeterminada. Cada fila representa una consulta omitida y la misma combinación puede aparecer más de una vez.

In [ ]:
auditoria = {"cambios_omitidos": [], "detecciones_ejecutadas": []}
simular_cache(eventos, pagos_observados, 12, auditoria=auditoria)

def clave_deuda(consulta):
    return (consulta.account_id, consulta.convenio, consulta.monto, consulta.fecha_vencimiento)

detecciones_ejecutadas = defaultdict(list)
for consulta in auditoria["detecciones_ejecutadas"]:
    detecciones_ejecutadas[clave_deuda(consulta)].append(consulta.timestamp)

filas_riesgo = []
for consulta in auditoria["cambios_omitidos"]:
    # Solo reportamos cambios omitidos durante semanas completas.
    if not date(2026, 8, 3) <= consulta.timestamp.date() <= date(2026, 10, 25):
        continue
    fechas = detecciones_ejecutadas[clave_deuda(consulta)]
    indice = bisect_right(fechas, consulta.timestamp)
    siguiente = fechas[indice] if indice < len(fechas) else None
    margen = (
        (consulta.fecha_vencimiento - siguiente.date()).days
        if siguiente is not None and consulta.fecha_vencimiento is not None else None
    )
    filas_riesgo.append({
        "cuenta": consulta.account_id,
        "convenio": consulta.convenio,
        "monto": consulta.monto,
        "vencimiento": consulta.fecha_vencimiento,
        "deteccion_omitida": consulta.timestamp,
        "siguiente_deteccion": siguiente,
        "demora_horas": (siguiente - consulta.timestamp).total_seconds() / 3600 if siguiente else None,
        "dias_hasta_vencimiento": margen,
        "estado": (
            "Sin detección posterior observada" if siguiente is None else
            "Vencimiento desconocido" if margen is None else
            "Después del vencimiento" if margen < 0 else
            "Día del vencimiento" if margen == 0 else "Antes del vencimiento"
        ),
    })

riesgo_cache = pd.DataFrame(filas_riesgo)
display(riesgo_cache.round(2))


## Paso 6 · Frecuencia observada por convenio

Medimos el intervalo entre consultas programadas consecutivas para la misma cuenta y convenio. El promedio diario y el porcentaje de días con varias consultas usan solo días en los que la cuenta tuvo al menos una consulta programada. El ratio del período sirve para comparar convenios. El límite de 6 se evalúa por semana para DeltaPay en conjunto.

Agua y TV concentran el 75,9 % de las consultas de DeltaPay. TV también concentra el 75,1 % de sus reintentos. Gas tiene el mayor ratio del período y 3.620 consultas programadas con resultado `sin_deuda` sobre 4.542 intentos totales. Luz registra una programada por día con actividad y un ratio de 1,94. Estas diferencias orientan qué convenios investigar antes de cambiar el calendario.

In [ ]:
consultar("""
WITH programadas AS (
    SELECT convenio, account_id, SUBSTR("timestamp", 1, 10) AS dia,
           CAST(SUBSTR("timestamp", 1, 19) AS TIMESTAMP) AS momento
    FROM consultas
    WHERE proveedor = 'deltapay' AND origen = 'programada'
),
por_dia AS (
    SELECT convenio, account_id, dia, COUNT(*) AS consultas
    FROM programadas
    GROUP BY 1, 2, 3
),
intervalos AS (
    SELECT convenio,
           momento - LAG(momento) OVER (
               PARTITION BY convenio, account_id ORDER BY momento
           ) AS intervalo
    FROM programadas
),
frecuencia AS (
    SELECT convenio, COUNT(*) AS cuenta_dias_con_consulta,
           AVG(consultas) AS programadas_por_cuenta_dia,
           COUNT(*) FILTER (WHERE consultas > 1) AS cuenta_dias_con_varias
    FROM por_dia GROUP BY convenio
),
medianas AS (
    SELECT convenio, MEDIAN(EPOCH(intervalo) / 3600) AS horas_entre_programadas
    FROM intervalos GROUP BY convenio
),
volumen AS (
    SELECT convenio, COUNT(*) AS consultas,
           COUNT(*) FILTER (WHERE origen = 'programada' AND resultado = 'sin_deuda') AS programadas_sin_deuda,
           COUNT(*) FILTER (WHERE origen = 'reintento') AS reintentos
    FROM consultas WHERE proveedor = 'deltapay' GROUP BY convenio
),
pagos_por_convenio AS (
    SELECT convenio, COUNT(*) AS pagos FROM pagos GROUP BY convenio
)
SELECT convenio, v.consultas, COALESCE(p.pagos, 0) AS pagos,
       ROUND(v.consultas * 1.0 / NULLIF(p.pagos, 0), 2) AS ratio_periodo,
       v.programadas_sin_deuda, v.reintentos,
       ROUND(f.programadas_por_cuenta_dia, 2) AS programadas_por_cuenta_dia,
       ROUND(100.0 * f.cuenta_dias_con_varias / f.cuenta_dias_con_consulta, 1) AS pct_cuenta_dias_con_varias,
       ROUND(m.horas_entre_programadas, 2) AS mediana_horas_entre_programadas
FROM volumen v
LEFT JOIN pagos_por_convenio p USING (convenio)
LEFT JOIN frecuencia f USING (convenio)
LEFT JOIN medianas m USING (convenio)
ORDER BY v.consultas DESC
""")

## Paso 7 · Espaciar consultas programadas en TV y Agua

Simulamos intervalos mínimos de 12 y 24 horas por cuenta y convenio entre consultas programadas que sí se ejecutan. Conservamos los reintentos y pagos observados. Algunos reintentos podrían depender de una consulta omitida, por eso la detección estimada puede ser optimista. Identificamos una deuda por cuenta, convenio y vencimiento como aproximación para seguir su primera detección. Los resultados corresponden a semanas completas.

In [ ]:
from statistics import median

def medir_frecuencia(horas, horas_agua=None, horas_agua_despues=None):
    intervalos = {
        "CONV-TV": timedelta(hours=horas),
        "CONV-AGUA": timedelta(hours=horas if horas_agua is None else horas_agua),
    }
    objetivos = {"CONV-TV", "CONV-AGUA"}
    ultima_programada = {}
    primera_observada = {}
    primera_con_regla = {}
    resumen = defaultdict(Counter)

    def lunes(momento):
        dia = momento.date()
        return dia - timedelta(days=dia.weekday())

    for consulta in eventos:
        momento = consulta.timestamp
        cuenta = (consulta.account_id, consulta.convenio)
        semana = lunes(momento)
        resumen[semana]["consultas"] += 1
        omitir = False
        if consulta.convenio in objetivos and consulta.origen == "programada":
            anterior = ultima_programada.get(cuenta)
            minimo = intervalos[consulta.convenio]
            if consulta.convenio == "CONV-AGUA" and horas_agua_despues is not None and momento.day > 10:
                minimo = timedelta(hours=horas_agua_despues)
            omitir = anterior is not None and momento - anterior < minimo
            if not omitir:
                ultima_programada[cuenta] = momento

        if omitir:
            resumen[semana]["omitidas"] += 1

        if consulta.resultado == "deuda_encontrada" and consulta.fecha_vencimiento is not None:
            clave = (*cuenta, consulta.fecha_vencimiento)
            primera_observada.setdefault(clave, momento)
            if not omitir:
                primera_con_regla.setdefault(clave, momento)

    for pago in pagos_observados:
        resumen[lunes(pago.fecha_pago)]["pagos"] += 1

    semanas_completas = [
        v for semana, v in sorted(resumen.items())
        if date(2026, 8, 3) <= semana <= date(2026, 10, 19)
    ]
    demoras = []
    riesgos = Counter()
    for clave, original in primera_observada.items():
        if clave[1] not in objetivos or not date(2026, 8, 3) <= original.date() <= date(2026, 10, 25):
            continue
        siguiente = primera_con_regla.get(clave)
        if siguiente is None:
            riesgos["sin_deteccion"] += 1
        elif siguiente > original:
            demoras.append((siguiente - original).total_seconds() / 3600)
            if siguiente.date() >= clave[-1] and original.date() < clave[-1]:
                riesgos["pierde_margen_previo"] += 1

    return {
        "horas_tv": horas,
        "horas_agua": horas if horas_agua is None else horas_agua,
        "horas_agua_despues": horas_agua_despues,
        "consultas_omitidas": sum(v["omitidas"] for v in semanas_completas),
        "semanas_en_limite": sum(
            v["pagos"] > 0 and v["consultas"] - v["omitidas"] <= 6 * v["pagos"]
            for v in semanas_completas
        ),
        "mejor_ratio": min((v["consultas"] - v["omitidas"]) / v["pagos"] for v in semanas_completas if v["pagos"]),
        "detecciones_demoradas": len(demoras),
        "mediana_demora_horas": median(demoras) if demoras else None,
        "sin_deteccion_observada": riesgos["sin_deteccion"],
        "pasa_al_dia_de_vencimiento_o_despues": riesgos["pierde_margen_previo"],
    }


display(pd.DataFrame([medir_frecuencia(h) for h in (12, 24)]).round(2))

## Paso 8 · Combinar las reglas de 12 horas

Simulamos ambos cambios sobre el mismo historial. Primero se evalúa si se puede reutilizar la deuda conocida. Para las consultas programadas de TV y Agua que siguen pendientes, exigimos un intervalo de 12 horas desde la última programada ejecutada. Una consulta omitida no actualiza el estado. Se conservan los pagos históricos y los reintentos siguen disponibles para la regla de reutilización aunque su consulta de origen se haya omitido. Identificamos la primera detección por cuenta, convenio y vencimiento, porque no hay ID de factura.

In [ ]:
auditoria_combinada = {
    "cambios_omitidos": [], "detecciones_ejecutadas": [],
    "primera_observada": {}, "primera_con_regla": {},
}
semanas_combinadas = simular_cache(
    eventos, pagos_observados, 12, auditoria=auditoria_combinada,
    intervalo_programado=timedelta(hours=12),
)
filas_combinadas = []
for semana, datos in sorted(semanas_combinadas.items()):
    if not date(2026, 8, 3) <= semana <= date(2026, 10, 19):
        continue
    consultas_restantes = datos["consultas"] - datos["omitidas"]
    pagos = datos["pagos"]
    filas_combinadas.append({
        "semana": semana, "consultas": datos["consultas"], "pagos": pagos,
        "omitidas": datos["omitidas"],
        "ratio_observado": datos["consultas"] / pagos if pagos else None,
        "ratio_combinado": consultas_restantes / pagos if pagos else None,
    })

riesgo_combinado = Counter()
for factura, original in auditoria_combinada["primera_observada"].items():
    vencimiento = factura[-1]
    if vencimiento is None or not date(2026, 8, 3) <= original.date() <= date(2026, 10, 25):
        continue
    deteccion = auditoria_combinada["primera_con_regla"].get(factura)
    if deteccion is None:
        riesgo_combinado["sin_deteccion_posterior"] += 1
    elif deteccion > original:
        riesgo_combinado["primeras_detecciones_demoradas"] += 1
        if original.date() < vencimiento <= deteccion.date():
            riesgo_combinado["pasan_al_vencimiento_o_despues"] += 1

detalle_combinado = pd.DataFrame(filas_combinadas)
display(detalle_combinado.round(2))
print("Consultas omitidas:", detalle_combinado["omitidas"].sum())
print("Semanas dentro del límite:", detalle_combinado["ratio_combinado"].le(6).sum())
for indicador in ("primeras_detecciones_demoradas", "sin_deteccion_posterior", "pasan_al_vencimiento_o_despues"):
    print(indicador, riesgo_combinado[indicador])


## Paso 9 · Probar un intervalo propio para Agua

Fijamos TV en 12 horas y comparamos 12, 18 y 24 horas para Agua, sin aplicar la regla de deuda guardada. Un intervalo mayor puede conservar consultas en otros horarios, por eso la cantidad de primeras detecciones demoradas no necesariamente crece de forma lineal. Se mantienen los pagos y reintentos históricos.

In [ ]:
display(pd.DataFrame([medir_frecuencia(12, horas) for horas in (12, 18, 24)]).round(2))

## Paso 10 · Combinar TV a 12 h, Agua a 18 h y deuda guardada a 12 h

La brecha semanal muestra cuántas consultas adicionales habría que evitar para llegar a 6 con los pagos observados. Se mantienen los reintentos históricos disponibles y se aproxima la factura por cuenta, convenio y vencimiento.

In [ ]:
auditoria_combinada = {
    "cambios_omitidos": [], "detecciones_ejecutadas": [],
    "primera_observada": {}, "primera_con_regla": {},
}
semanas_combinadas = simular_cache(
    eventos, pagos_observados, 12, auditoria=auditoria_combinada,
    intervalo_programado={'CONV-TV': timedelta(hours=12), 'CONV-AGUA': timedelta(hours=18)},
)
filas_combinadas = []
for semana, datos in sorted(semanas_combinadas.items()):
    if not date(2026, 8, 3) <= semana <= date(2026, 10, 19):
        continue
    consultas_restantes = datos["consultas"] - datos["omitidas"]
    pagos = datos["pagos"]
    filas_combinadas.append({
        "semana": semana, "consultas": datos["consultas"], "pagos": pagos,
        "omitidas": datos["omitidas"],
        "consultas_a_reducir": max(consultas_restantes - 6 * pagos, 0),
        "ratio_observado": datos["consultas"] / pagos if pagos else None,
        "ratio_combinado": consultas_restantes / pagos if pagos else None,
    })

riesgo_combinado = Counter()
for factura, original in auditoria_combinada["primera_observada"].items():
    vencimiento = factura[-1]
    if vencimiento is None or not date(2026, 8, 3) <= original.date() <= date(2026, 10, 25):
        continue
    deteccion = auditoria_combinada["primera_con_regla"].get(factura)
    if deteccion is None:
        riesgo_combinado["sin_deteccion_posterior"] += 1
    elif deteccion > original:
        riesgo_combinado["primeras_detecciones_demoradas"] += 1
        if original.date() < vencimiento <= deteccion.date():
            riesgo_combinado["pasan_al_vencimiento_o_despues"] += 1

detalle_combinado = pd.DataFrame(filas_combinadas)
display(detalle_combinado.round(2))
print("Consultas omitidas:", detalle_combinado["omitidas"].sum())
print("Semanas dentro del límite:", detalle_combinado["ratio_combinado"].le(6).sum())
for indicador in ("primeras_detecciones_demoradas", "sin_deteccion_posterior", "pasan_al_vencimiento_o_despues"):
    print(indicador, riesgo_combinado[indicador])


## Paso 11 · Calendario de pagos y vencimientos

Cruzamos la semana de pago con la semana de vencimiento de las facturas pagadas. El corte mensual por días 1 a 7 permite comparar el comienzo de cada mes con la misma ventana. El CSV de pagos solo muestra vencimientos de facturas pagadas.

In [ ]:
consultar("""
SELECT convenio,
       date_trunc('week', CAST(substr(fecha_pago,1,10) AS DATE)) AS semana_pago,
       date_trunc('week', CAST(fecha_vencimiento AS DATE)) AS semana_vencimiento,
       COUNT(*) AS pagos
FROM pagos
WHERE convenio IN (SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay')
GROUP BY 1, 2, 3
ORDER BY 2, 1, 3
""")

In [ ]:
consultar("""
SELECT substr(fecha_pago, 1, 7) AS mes,
       COUNT(*) AS pagos,
       COUNT(*) FILTER (WHERE substr(fecha_pago, 9, 2) <= '07') AS pagos_dias_1_a_7,
       ROUND(100.0 * pagos_dias_1_a_7 / COUNT(*), 1) AS pct_dias_1_a_7
FROM pagos
WHERE convenio IN (SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay')
GROUP BY 1
ORDER BY 1
""")

## Paso 12 · Picos de pago y detección por convenio

Agrupamos por día del mes y mostramos los tres mayores conteos, incluidos los empates. La primera detección se aproxima por cuenta, convenio y vencimiento. El día en que aparece en los datos depende del calendario histórico de consultas. Usamos los picos para definir ventanas candidatas y evaluar su impacto en una simulación.

In [ ]:
consultar("""
WITH convenios_delta AS (
    SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay'
),
primeras_detecciones AS (
    SELECT convenio, account_id, fecha_vencimiento,
           MIN(SUBSTR("timestamp", 1, 10)) AS fecha
    FROM consultas
    WHERE proveedor = 'deltapay' AND resultado = 'deuda_encontrada'
      AND fecha_vencimiento IS NOT NULL
    GROUP BY 1, 2, 3
),
eventos_por_dia AS (
    SELECT convenio, SUBSTR(fecha_pago, 1, 10) AS fecha, 'Pago' AS evento
    FROM pagos WHERE convenio IN (SELECT convenio FROM convenios_delta)
    UNION ALL
    SELECT convenio, fecha, 'Primera detección' FROM primeras_detecciones
),
conteos AS (
    SELECT convenio, evento, SUBSTR(fecha, 9, 2) AS dia_del_mes,
           COUNT(*) AS cantidad
    FROM eventos_por_dia
    GROUP BY 1, 2, 3
)
SELECT *, DENSE_RANK() OVER (
    PARTITION BY convenio, evento ORDER BY cantidad DESC
) AS puesto
FROM conteos
QUALIFY puesto <= 3
ORDER BY convenio, evento, puesto, dia_del_mes
""")

## Paso 13 · Frecuencia de Agua por día del mes

Comparamos 18 h fijas con 18 h del día 1 al 10 y 24 h desde el día 11, usando el día local de cada consulta programada. TV queda a 12 h en ambos escenarios. Conservamos los pagos y reintentos observados. La primera detección se aproxima por cuenta, convenio y vencimiento.

In [ ]:
display(pd.DataFrame([medir_frecuencia(12, 18), medir_frecuencia(12, 18, 24)],
                     index=["18 h fijas", "18 h hasta el día 10, luego 24 h"]).round(2))